In [1]:
# change to the root directory of the project
import os
if os.getcwd().split("/")[-1] == "examples":
    os.chdir('..')
print(os.getcwd())

import os
import h5py
import pandas as pd


/mnt/raid/data/anina/ScanDy


### Read data from hdf file

In [2]:
def load_evolution_from_hdf5(hdf_path, param_names=None, max_generations=100):
    """
    Load evolution data from neurolib HDF5 file into a pandas DataFrame.
    
    Parameters
    ----------
    hdf_path : str
        Path to the HDF5 file
    param_names : list of str, optional
        Names of the parameters. If None, will use generic names.
    max_generations : int, optional
        Maximum number of generations to try loading (default: 100)
    
    Returns
    -------
    df_evol : pandas.DataFrame
        DataFrame with all individuals from all generations
    """
    
    with h5py.File(hdf_path, 'r') as f:
        # Get the trajectory name (should be the first key)
        traj_name = list(f.keys())[0]
        
        all_data = []
        
        for gen_num in range(max_generations):
            gen_key = f"gen_{gen_num:06d}"
            
            gen_path = f"{traj_name}/results/evolution/{gen_key}"
            if gen_key not in f[f"{traj_name}/results/evolution"]:
                break
                
            # Read population (parameters)
            pop = f[f"{gen_path}/population/population"][:]
            
            # Read scores
            scores = f[f"{gen_path}/scores/scores"][:]
            
            # Read fitness (multi-objective values)
            fitness = f[f"{gen_path}/fitness/fitness"][:]
            
            # Determine how many parameters to extract
            if param_names is not None:
                num_params = len(param_names)
            else:
                num_params = pop.shape[1]
            

            for i in range(len(pop)):
                row = list(pop[i][:num_params]) + [gen_num, scores[i]] + list(fitness[i])
                all_data.append(row)
        
        if param_names is None:
            param_names = [f"param_{i}" for i in range(num_params)]
        
        num_fitness = fitness.shape[1]
        fitness_cols = [f"fitness_{i}" for i in range(num_fitness)]
        columns = param_names + ["generation", "score"] + fitness_cols
        
        df_evol = pd.DataFrame(all_data, columns=columns)
        
    return df_evol


In [3]:
hdf_path = "/mnt/raid/data/anina/ScanDy/data/hdf/new_fit/evol_bdir_obj_ll_cb_s13__std1_seed10_young.hdf"
param_names = ["ddm_thres", "ddm_sig", "att_dva", "ior_decay", "ior_inobj"]

df_evol = load_evolution_from_hdf5(hdf_path, param_names=param_names)
df_evol.sort_values("score", ascending=False)

,ddm_thres,ddm_sig,att_dva,ior_decay,ior_inobj,generation,score,fitness_0,fitness_1,fitness_2,fitness_3
1200,0.638007,0.061498,89.988438,991.397139,0.929633,29,-0.223161,0.150525,0.099807,0.133398,0.143289
1160,0.823845,0.052813,53.890527,967.692427,0.903877,28,-0.227532,0.152858,0.082858,0.142143,0.141590
1120,0.823845,0.052813,53.890527,967.692427,0.903877,27,-0.227532,0.152858,0.082858,0.142143,0.141590
1201,0.823845,0.052813,53.890527,967.692427,0.903877,29,-0.227532,0.152858,0.082858,0.142143,0.141590
1161,0.729232,0.050425,79.609603,985.565053,0.897700,28,-0.231161,0.256955,0.099106,0.139236,0.139307
...,...,...,...,...,...,...,...,...,...,...,...
1,5.607149,0.073049,24.118629,425.533932,0.958084,0,-0.762669,0.571716,0.272784,0.583396,0.363121
35,4.534493,0.060687,25.338238,956.954136,0.976567,0,-0.763404,0.521471,0.305521,0.578542,0.370414
29,0.852062,0.134141,4.921673,385.873854,0.101029,0,-0.764020,0.886471,0.406447,0.552252,0.358698
58,5.718210,0.087714,54.950081,200.673308,0.997519,0,-0.832567,0.589905,0.336723,0.645826,0.387044
